# 01 · Vectors, dot products and similarity

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/math/01_vectors_similarity.ipynb) &nbsp; 📖 Concept deck: [Vectors and dot products](https://nilesh-salpe.github.io/visual-ai/math/01-vectors-dot-products/) &nbsp; ⏱️ Suggested time: **25 min**

**You will build:** dot products, norms and cosine similarity from scratch, a nearest-neighbour search over toy embeddings, and the √d scaling that attention uses.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU with NumPy, so no GPU is needed.

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")


rng = np.random.default_rng(0)
WORDS = ['king', 'queen', 'man', 'woman', 'apple', 'banana']
# toy 4-d embeddings: [royalty, male, female, fruit]
EMB = np.array([[0.9, 0.8, 0.1, 0.0], [0.9, 0.1, 0.8, 0.0], [0.1, 0.9, 0.1, 0.0],
                [0.1, 0.1, 0.9, 0.0], [0.0, 0.0, 0.1, 0.9], [0.0, 0.1, 0.0, 0.95]])

## Task 1: Cosine similarity

Implement $\cos(a, b) = \dfrac{a \cdot b}{\lVert a \rVert\,\lVert b \rVert}$ using only `np.sum` and `np.sqrt` (no `np.dot`, no `np.linalg`).
It is the dot product after both vectors are scaled to length 1, so it measures **direction only**.

In [ ]:
def cosine(a, b):
    # TODO: dot product divided by the product of the two lengths
    raise NotImplementedError

In [ ]:
# @title Tests for Task 1: run after implementing
def _test_cosine(f):
    assert np.isclose(f(np.array([1., 0]), np.array([0., 1])), 0), "orthogonal vectors have cosine 0"
    assert np.isclose(f(np.array([1., 2]), np.array([2., 4])), 1), "parallel vectors have cosine 1"
    assert np.isclose(f(np.array([1., 2]), np.array([-1., -2])), -1), "opposite vectors have cosine -1"
    a, b = rng.normal(size=8), rng.normal(size=8)
    assert np.isclose(f(a, b), a @ b / np.linalg.norm(a) / np.linalg.norm(b))
    assert np.isclose(f(3 * a, b), f(a, b)), "cosine must ignore length"

check("cosine", _test_cosine, cosine)

## Task 2: Nearest neighbours by cosine

Given a query vector and a matrix of embeddings (one row per word), return the indices of the `k` most similar rows, most similar first.
Do it **vectorised**: normalise every row once, then one matrix-vector product gives all cosines at once. This is exactly how embedding search works.

In [ ]:
def nearest(query, E, k=2):
    # TODO: normalise rows of E and the query, compute all cosines with one matmul, return top-k indices
    raise NotImplementedError

In [ ]:
# @title Tests for Task 2: run after implementing
def _test_nearest(f):
    idx = list(f(EMB[0], EMB, k=2))
    assert idx[0] == 0, "the closest vector to 'king' is 'king' itself"
    assert WORDS[idx[1]] in ('queen', 'man'), f"second nearest to king should be queen or man, got {WORDS[idx[1]]}"
    target = EMB[0] - EMB[2] + EMB[3]                    # king - man + woman
    best = [WORDS[i] for i in f(target, EMB, k=3)]
    assert best[0] == 'queen', f"king - man + woman should land nearest to queen, got {best}"
    assert WORDS[list(f(EMB[4], EMB, 2))[1]] == "banana"

check("nearest", _test_nearest, nearest)

## Task 3: Why attention divides by √d

Draw `n` pairs of random vectors with independent entries of mean 0 and variance 1 in dimension `d`, and return the **standard deviation of their dot products**.
The Transformer paper notes that such a dot product has variance $d$, so its spread grows like $\sqrt{d}$. Dividing scores by $\sqrt{d}$ keeps the softmax out of its flat, saturated regions.

In [ ]:
def dot_std(d, n=20000, seed=0):
    r = np.random.default_rng(seed)
    # TODO: n pairs of d-dim standard-normal vectors; std of their dot products
    raise NotImplementedError

In [ ]:
# @title Tests for Task 3: run after implementing
def _test_dot_std(f):
    for d in [4, 64, 256]:
        s = f(d)
        assert abs(s / np.sqrt(d) - 1) < 0.05, f"d={d}: expected std ≈ √d = {np.sqrt(d):.1f}, got {s:.2f}"

check("dot_std", _test_dot_std, dot_std)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: Cosine similarity
def cosine_solution(a, b):
    return np.sum(a * b) / (np.sqrt(np.sum(a * a)) * np.sqrt(np.sum(b * b)))

check("cosine (reference)", _test_cosine, cosine_solution)

In [ ]:
# @title Solution for Task 2: Nearest neighbours by cosine
def nearest_solution(query, E, k=2):
    En = E / np.linalg.norm(E, axis=1, keepdims=True)
    sims = En @ (query / np.linalg.norm(query))
    return np.argsort(-sims)[:k]

check("nearest (reference)", _test_nearest, nearest_solution)

In [ ]:
# @title Solution for Task 3: Why attention divides by √d
def dot_std_solution(d, n=20000, seed=0):
    r = np.random.default_rng(seed)
    a, b = r.normal(size=(n, d)), r.normal(size=(n, d))
    return np.sum(a * b, axis=1).std()

check("dot_std (reference)", _test_dot_std, dot_std_solution)

---
## Experiments
**Random vectors in high dimension are nearly orthogonal.** The cosine of two random directions concentrates around 0 with spread ≈ 1/√d, which is why a few thousand dimensions can hold many nearly independent "features".

In [ ]:
r = np.random.default_rng(1)
for d in [2, 16, 128, 1024, 4096]:
    a, b = r.normal(size=(5000, d)), r.normal(size=(5000, d))
    cos = np.sum(a * b, 1) / np.linalg.norm(a, axis=1) / np.linalg.norm(b, axis=1)
    print(f"d={d:5d}: std of cos = {cos.std():.3f}   (1/√d = {1 / np.sqrt(d):.3f})")

---
## Takeaways
* The dot product measures alignment scaled by length; cosine similarity removes the length.
* Embedding search is one normalised matrix-vector product followed by a top-k.
* Dot products of random d-dimensional vectors have spread √d, hence the 1/√d in scaled dot-product attention.